Install dependencies (FIRST always)

In [1]:
!pip install mediapipe==0.10.13 scikit-learn opencv-python

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.6/35.6 MB 39.0 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 294.9/294.9 kB 13.5 MB/s eta 0:00:00
  Attempting uninstall: protobuf
    Found existing installation: protobuf 5.29.5
    Uninstalling protobuf-5.29.5:
      Successfully uninstalled protobuf-5.29.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.31.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-adk 1.21.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
a2a-sdk 0.3.23 requires protobuf>=5.29.5, but you have protobuf 4.25.8 which is incompatible.
grain 0.2.15 requires protobuf>=5.28.3, but you have protobuf 4.25.8 which is incompatible.
opentelemetry-proto 1.37.0 requires protobuf<7.0,>=5.0, but you have protobuf 4.25.8 which is incompatible.
ydf 0.14.0 r

Imports

In [2]:
import os
import cv2
import mediapipe as mp
import numpy as np
import pandas as pd

2026-03-21 16:24:50.929755: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1774110291.183898      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1774110291.253055      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1774110291.863027      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774110291.863082      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1774110291.863085      55 computation_placer.cc:177] computation placer alr

Initialize MediaPipe

In [13]:
mp_pose = mp.solutions.pose
mp_drawing = mp.solutions.drawing_utils

pose = mp_pose.Pose()

print("MediaPipe Pose model loaded successfully")

MediaPipe Pose model loaded successfully


Debug

In [4]:
print(dir(mp))

['CalculatorGraph', 'GraphInputStreamAddMode', 'Image', 'ImageFormat', 'ImageFrame', 'Matrix', 'Packet', 'Timestamp', 'ValidatedGraphConfig', '__builtins__', '__cached__', '__doc__', '__file__', '__loader__', '__name__', '__package__', '__path__', '__spec__', '__version__', 'calculators', 'model_ckpt_util', 'packet_creator', 'packet_getter', 'resource_util', 'solutions', 'tasks']


Dataset Path

In [8]:
dataset_path = "/kaggle/input/datasets/khushwantmehra/yoga-video-dataset/Final_project3_dataset"

Load Video Paths + Labels

In [9]:
import os

video_files = []
labels = []

for pose_name in os.listdir(dataset_path):

    pose_path = os.path.join(dataset_path, pose_name)

    if not os.path.isdir(pose_path):
        continue

    for quality in os.listdir(pose_path):

        quality_path = os.path.join(pose_path, quality)

        if not os.path.isdir(quality_path):
            continue

        for file in os.listdir(quality_path):

            if file.endswith(".mp4"):

                video_files.append(os.path.join(quality_path, file))
                labels.append(pose_name + "_" + quality)

print("Total videos:", len(video_files))
print("Example:", video_files[0])

Total videos: 461
Example: /kaggle/input/datasets/khushwantmehra/yoga-video-dataset/Final_project3_dataset/trikonasana/avg/20260221_081745.mp4


Landmark Extraction Function

In [11]:
def extract_landmarks(frame):

    image = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = pose.process(image)

    if results.pose_landmarks:

        lm = results.pose_landmarks.landmark

        coords = []
        for point in lm:
            coords.extend([point.x, point.y, point.z])

        coords = np.array(coords).reshape(33, 3)

        # 🔥 NORMALIZATION (IMPORTANT)
        center = coords[23]  # LEFT_HIP
        coords = coords - center

        max_value = np.max(np.abs(coords))
        if max_value != 0:
            coords = coords / max_value

        # 🔥 ANGLE FUNCTION
        def calculate_angle(a, b, c):
            a = np.array(a)
            b = np.array(b)
            c = np.array(c)

            radians = np.arctan2(c[1]-b[1], c[0]-b[0]) - \
                      np.arctan2(a[1]-b[1], a[0]-b[0])

            angle = np.abs(radians * 180.0 / np.pi)

            if angle > 180:
                angle = 360 - angle

            return angle

        # 🔥 ANGLES
        shoulder = coords[11][:2]
        elbow = coords[13][:2]
        wrist = coords[15][:2]

        hip = coords[23][:2]
        knee = coords[25][:2]
        ankle = coords[27][:2]

        elbow_angle = calculate_angle(shoulder, elbow, wrist)
        knee_angle = calculate_angle(hip, knee, ankle)

        # 🔥 NEW FEATURES (VERY IMPORTANT)

        # 1️⃣ Ankle distance (Tadasana vs Vrikshasana)
        left_ankle = coords[27][:2]
        right_ankle = coords[28][:2]
        ankle_distance = np.linalg.norm(
            np.array(left_ankle) - np.array(right_ankle)
        )

        # 2️⃣ Knee height difference (one leg lifted or not)
        left_knee_y = coords[25][1]
        right_knee_y = coords[26][1]
        knee_height_diff = abs(left_knee_y - right_knee_y)

        # 3️⃣ Hip alignment (body balance)
        left_hip = coords[23][:2]
        right_hip = coords[24][:2]
        hip_distance = np.linalg.norm(
            np.array(left_hip) - np.array(right_hip)
        )

        # 🔥 FINAL FEATURE VECTOR
        features = coords.flatten().tolist() + [
            elbow_angle,
            knee_angle,
            ankle_distance,
            knee_height_diff,
            hip_distance
        ]

        return features

    return None

Extract Data from Videos

In [14]:
data = []
data_labels = []

for video, label in zip(video_files, labels):

    cap = cv2.VideoCapture(video)

    frame_count = 0

    while cap.isOpened():

        ret, frame = cap.read()

        if not ret:
            break

        frame_count += 1

        if frame_count % 5 != 0:
            continue

        landmarks = extract_landmarks(frame)

        if landmarks:
            data.append(landmarks)
            data_labels.append(label)

    cap.release()

print("Total pose samples:", len(data))

/usr/local/lib/python3.12/dist-packages/google/protobuf/symbol_database.py:55: UserWarning: SymbolDatabase.GetPrototype() is deprecated. Please use message_factory.GetMessageClass() instead. SymbolDatabase.GetPrototype() will be removed soon.
  warnings.warn('SymbolDatabase.GetPrototype() is deprecated. Please '


Total pose samples: 13847


Convert to Dataset

In [15]:
X = pd.DataFrame(data)
y = np.array(data_labels)

print("Feature shape:", X.shape)

Feature shape: (13847, 104)


Train-Test Split

In [16]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42)

Train Model

In [17]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=300, class_weight='balanced')
model.fit(X_train, y_train)

RandomForestClassifier(class_weight='balanced', n_estimators=300)

Evaluate Model

In [18]:
from sklearn.metrics import accuracy_score

pred = model.predict(X_test)
accuracy = accuracy_score(y_test, pred)

print("Model Accuracy:", accuracy)

Model Accuracy: 0.8812274368231047


Save Model

In [21]:
import joblib

# Save (compressed)
joblib.dump(model, "yoga_model_2.pkl", compress=("lzma", 9))

['yoga_model_2.pkl']